In [33]:
# ============================================================
# [CELL 1] MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [34]:
# ============================================================
# [CELL 2] SETUP - Cai dat thu vien can thiet
# ------------------------------------------------------------
# sqlite3: thu vien chuan cua Python, khong can cai dat them,
#          dung de tao va thao tac co so du lieu SQLite.
# pandas: doc file ket qua dem va ho tro hien thi ket qua
#         truy van duoi dang bang de de doi chieu.
# ============================================================
import sqlite3
import pandas as pd
import os
from datetime import datetime, timedelta

print("Da import xong thu vien.")
print("Phien ban sqlite3:", sqlite3.sqlite_version)


Da import xong thu vien.
Phien ban sqlite3: 3.37.2


In [35]:
# ============================================================
# [CELL 3] PATH - Khai bao duong dan file nguon va file CSDL
# ------------------------------------------------------------
# CSV_NGUON: ket qua dem line-crossing da tao o muc 3.4.2,
#            dung lam nguon du lieu dau vao cho CSDL.
# DB_PATH  : duong dan file co so du lieu SQLite se duoc tao,
#            day la 1 file duy nhat, khong can cai dat may chu.
# ============================================================
OUT_DIR   = "/content/drive/MyDrive/Do an AI giao thong/Tracking-LineCrossing"
CSV_NGUON = os.path.join(OUT_DIR, "ket_qua_dem_line_crossing.csv")

DB_DIR  = "/content/drive/MyDrive/Do an AI giao thong/SQLite/Database"
os.makedirs(DB_DIR, exist_ok=True)
DB_PATH = os.path.join(DB_DIR, "he_thong_dem_xe.db")

assert os.path.exists(CSV_NGUON), f"Khong thay file: {CSV_NGUON}"

df_nguon = pd.read_csv(CSV_NGUON)
print("Da doc file nguon. Tong so dong:", len(df_nguon))
print("Cac cot hien co:", list(df_nguon.columns))
print(df_nguon.head())

Da doc file nguon. Tong so dong: 6
Cac cot hien co: ['timestamp', 'track_id', 'class', 'chieu', 'frame_cat_vach']
             timestamp  track_id     class                            chieu  \
0  2026-09-01 09:58:55         2       bus  Huong di Duong Nguyen Chi Thanh   
1  2026-09-01 09:58:54        22  motobike         Huong di Duong 3 Thang 2   
2  2026-09-01 09:58:56        27  motobike  Huong di Duong Nguyen Chi Thanh   
3  2026-09-01 09:58:57        29  motobike  Huong di Duong Nguyen Chi Thanh   
4  2026-09-01 09:58:57        35  motobike  Huong di Duong Nguyen Chi Thanh   

   frame_cat_vach  
0              60  
1              24  
2              72  
3              96  
4             117  


In [36]:
# ============================================================
# [CELL 4] TAO CO SO DU LIEU - Thiet ke bang lich_su_dem
# ------------------------------------------------------------
# Bang duy nhat luu du lieu goc, moi dong tuong ung 1 luot
# phuong tien cu the di qua vach cat, kem day du thoi gian
# chinh xac. Day la TANG 1 trong kien truc hai tang da chot.
#
# Cau truc bang:
#   id               : ma so tu dong tang, khoa chinh
#   thoi_gian        : ngay gio day du (YYYY-MM-DD HH:MM:SS)
#   loai_phuong_tien : ten lop (car, motorbike, truck, bus)
#   chieu_di_chuyen  : huong di chuyen
#   track_id_goc     : ma dinh danh goc tu ByteTrack, phuc vu
#                      doi chieu va debug trong giai doan
#                      thuc nghiem
# ============================================================
ket_noi = sqlite3.connect(DB_PATH)
con_tro = ket_noi.cursor()

con_tro.execute("DROP TABLE IF EXISTS lich_su_dem")
con_tro.execute("""
    CREATE TABLE lich_su_dem (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        thoi_gian        TEXT NOT NULL,
        loai_phuong_tien TEXT NOT NULL,
        chieu_di_chuyen  TEXT NOT NULL,
        track_id_goc     INTEGER
    )
""")
ket_noi.commit()
print("Da tao bang lich_su_dem thanh cong.")

con_tro.execute("PRAGMA table_info(lich_su_dem)")
print("")
print("Cau truc bang:")
for cot in con_tro.fetchall():
    ten_cot = str(cot[1]).ljust(20)
    kieu_du_lieu = str(cot[2])
    print("  " + ten_cot + " | kieu du lieu: " + kieu_du_lieu)

Da tao bang lich_su_dem thanh cong.

Cau truc bang:
  id                   | kieu du lieu: INTEGER
  thoi_gian            | kieu du lieu: TEXT
  loai_phuong_tien     | kieu du lieu: TEXT
  chieu_di_chuyen      | kieu du lieu: TEXT
  track_id_goc         | kieu du lieu: INTEGER


In [37]:
# ============================================================
# [CELL 5] GHI DU LIEU - Nap du lieu tu CSV vao bang, su dung
#          DUNG THOI GIAN THUC TE co san trong du lieu nguon
# ------------------------------------------------------------
# Truoc day cell nay doc CSV va gia dinh chia du lieu thanh 2
# ngay khac nhau (29/08 va 30/08) chi de minh hoa kha nang truy
# van theo nhieu ngay. Hien tai df_nguon (doc tu vehicle_events.csv)
# da co san cot "timestamp" ghi nhan dung thoi diem he thong chay
# chuong trinh demo (tinh tu Cell 7 - module line-crossing, dua
# tren frame_cat_vach va FPS thuc te). Vi vay cell nay CHI can
# doc va nap dung 1 lan, KHONG gia dinh hay nhan ban du lieu
# theo nhieu ngay nua.
# ============================================================

tong_so_dong_da_ghi = 0

for _, dong in df_nguon.iterrows():
    con_tro.execute("""
        INSERT INTO lich_su_dem (thoi_gian, loai_phuong_tien, chieu_di_chuyen, track_id_goc)
        VALUES (?, ?, ?, ?)
    """, (
        dong["timestamp"],
        dong["class"],
        dong["chieu"],
        int(dong["track_id"]),
    ))
    tong_so_dong_da_ghi += 1

ket_noi.commit()

print(f"Da ghi tong cong {tong_so_dong_da_ghi} dong vao bang lich_su_dem.")
print(f"(tuong ung {len(df_nguon)} luot dem thuc te tu vehicle_events.csv)")
print(f"Khoang thoi gian du lieu: {df_nguon['timestamp'].min()} -> {df_nguon['timestamp'].max()}")

Da ghi tong cong 6 dong vao bang lich_su_dem.
(tuong ung 6 luot dem thuc te tu vehicle_events.csv)
Khoang thoi gian du lieu: 2026-09-01 09:58:54 -> 2026-09-01 09:58:58


In [38]:
# ============================================================
# [CELL 6] KIEM TRA DU LIEU - Xem thu vai dong da ghi
# ------------------------------------------------------------
# Doc lai truc tiep tu file CSDL (khong phai tu bien trong
# Python) de xac nhan dung du lieu da duoc luu ben trong file
# SQLite, dam bao viec ghi da thanh cong that su.
# ============================================================
kiem_tra = pd.read_sql_query("SELECT * FROM lich_su_dem ORDER BY thoi_gian LIMIT 10", ket_noi)
print("10 dong dau tien trong bang lich_su_dem:")
print(kiem_tra.to_string(index=False))

tong_so = pd.read_sql_query("SELECT COUNT(*) as tong FROM lich_su_dem", ket_noi)
print(f"\nTong so dong hien co trong bang: {tong_so['tong'][0]}")

10 dong dau tien trong bang lich_su_dem:
 id           thoi_gian loai_phuong_tien                 chieu_di_chuyen  track_id_goc
  2 2026-09-01 09:58:54         motobike        Huong di Duong 3 Thang 2            22
  1 2026-09-01 09:58:55              bus Huong di Duong Nguyen Chi Thanh             2
  3 2026-09-01 09:58:56         motobike Huong di Duong Nguyen Chi Thanh            27
  4 2026-09-01 09:58:57         motobike Huong di Duong Nguyen Chi Thanh            29
  5 2026-09-01 09:58:57         motobike Huong di Duong Nguyen Chi Thanh            35
  6 2026-09-01 09:58:58         motobike Huong di Duong Nguyen Chi Thanh            41

Tong so dong hien co trong bang: 6


In [39]:
# ============================================================
# [CELL 7] TRUY VAN MAU 1 - Tong hop theo Ngay x Loai phuong
#          tien, tach rieng theo tung chieu
# ------------------------------------------------------------
# Day la TANG 2 trong kien truc da chot: khong luu san, ma
# tinh truc tiep tu bang goc lich_su_dem bang cau lenh SQL.
# ============================================================
truy_van_1 = pd.read_sql_query("""
    SELECT
        DATE(thoi_gian) AS ngay,
        loai_phuong_tien,
        SUM(CASE WHEN chieu_di_chuyen LIKE '%3 Thang 2%' THEN 1 ELSE 0 END) AS huong_3_2,
        SUM(CASE WHEN chieu_di_chuyen LIKE '%Nguyen Chi Thanh%' THEN 1 ELSE 0 END) AS huong_nguyen_chi_thanh
    FROM lich_su_dem
    GROUP BY ngay, loai_phuong_tien
    ORDER BY ngay, loai_phuong_tien
""", ket_noi)

print("Bang tong hop theo Ngay va Loai phuong tien:")
print(truy_van_1.to_string(index=False))

Bang tong hop theo Ngay va Loai phuong tien:
      ngay loai_phuong_tien  huong_3_2  huong_nguyen_chi_thanh
2026-09-01              bus          0                       1
2026-09-01         motobike          1                       4


In [40]:
# ============================================================
# [CELL 8] TRUY VAN MAU 2 - Tong so luong theo mot khoang
#          thoi gian tuy y (nhap truc tiep khi chay cell)
# ------------------------------------------------------------
# Khi chay cell nay, chuong trinh se hoi nguoi dung nhap:
#   - Thoi diem bat dau: ngay/thang/nam gio:phut:giay
#   - Thoi diem ket thuc: ngay/thang/nam gio:phut:giay
# Dinh dang nhap: dd/mm/yyyy HH:MM:SS
# Vi du: 29/08/2026 07:00:00
#
# Cell nay cung tu dong mo lai ket noi CSDL neu ket noi truoc
# do da bi dong (con_noi.close()), tranh loi
# "Cannot operate on a closed database".
# ============================================================

import sqlite3
from datetime import datetime

# --- Tu dong mo lai ket noi neu da bi dong ---
try:
    ket_noi.execute("SELECT 1")
except (sqlite3.ProgrammingError, sqlite3.OperationalError, NameError):
    ket_noi = sqlite3.connect(DB_PATH)
    print("Da mo lai ket noi CSDL.")

# --- Nhap khoang thoi gian can truy van ---
DINH_DANG = "%d/%m/%Y %H:%M:%S"

while True:
    try:
        chuoi_bat_dau = input("Nhap thoi diem BAT DAU (dd/mm/yyyy HH:MM:SS): ").strip()
        thoi_gian_bat_dau_dt = datetime.strptime(chuoi_bat_dau, DINH_DANG)
        break
    except ValueError:
        print("Sai dinh dang. Vui long nhap dung mau: 29/08/2026 07:00:00")

while True:
    try:
        chuoi_ket_thuc = input("Nhap thoi diem KET THUC (dd/mm/yyyy HH:MM:SS): ").strip()
        thoi_gian_ket_thuc_dt = datetime.strptime(chuoi_ket_thuc, DINH_DANG)
        break
    except ValueError:
        print("Sai dinh dang. Vui long nhap dung mau: 30/08/2026 23:59:59")

thoi_gian_bat_dau = thoi_gian_bat_dau_dt.strftime("%Y-%m-%d %H:%M:%S")
thoi_gian_ket_thuc = thoi_gian_ket_thuc_dt.strftime("%Y-%m-%d %H:%M:%S")

# --- Thuc hien truy van ---
truy_van_2 = pd.read_sql_query("""
    SELECT
        loai_phuong_tien,
        SUM(CASE WHEN chieu_di_chuyen LIKE '%3 Thang 2%' THEN 1 ELSE 0 END) AS huong_3_2,
        SUM(CASE WHEN chieu_di_chuyen LIKE '%Nguyen Chi Thanh%' THEN 1 ELSE 0 END) AS huong_nguyen_chi_thanh,
        COUNT(*) AS tong_ca_2_huong
    FROM lich_su_dem
    WHERE thoi_gian BETWEEN ? AND ?
    GROUP BY loai_phuong_tien
    ORDER BY loai_phuong_tien
""", ket_noi, params=(thoi_gian_bat_dau, thoi_gian_ket_thuc))

print(f"\nTong hop tu {thoi_gian_bat_dau} den {thoi_gian_ket_thuc}:")

if len(truy_van_2) > 0:
    print(truy_van_2.to_string(index=False))
else:
    print("Khong co du lieu nao trong khoang thoi gian nay.")

Nhap thoi diem BAT DAU (dd/mm/yyyy HH:MM:SS): 01/09/2026 09:58:54
Nhap thoi diem KET THUC (dd/mm/yyyy HH:MM:SS): 01/09/2026 09:58:56

Tong hop tu 2026-09-01 09:58:54 den 2026-09-01 09:58:56:
loai_phuong_tien  huong_3_2  huong_nguyen_chi_thanh  tong_ca_2_huong
             bus          0                       1                1
        motobike          1                       1                2
